In [1]:
import datetime as dt
import json
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import seaborn as sns

# Setup file paths
logs_dir = "../dataset/"

# List of target JSON files to process
json_files_to_process = [
    "xgb_gss_results.json",
    "xgb_gss_val_results.json",
    "xgb_permutation_results.json",  # Supported for backward compatibility
]

dataset_keys = [
    ("raw_data_results", "Raw Data"),
    ("pruned_data_results", "Pruned Data"),
    ("pca_data_results", "PCA Data"),
]


def extract_metrics(ds_data):
    """Safely extracts metric dict regardless of whether it's under 'metrics' or 'final_results'."""
    if not isinstance(ds_data, dict):
        return {}
    if "metrics" in ds_data and ds_data["metrics"] is not None:
        return ds_data["metrics"]
    if "final_results" in ds_data and ds_data["final_results"] is not None:
        return ds_data["final_results"]
    return {}


def render_confusion_matrix(ax, cm_dict, title):
    """Reconstructs and plots a normalized confusion matrix heatmap."""
    if not cm_dict:
        ax.text(0.5, 0.5, "No Data", ha="center", va="center", fontsize=10)
        ax.set_title(title, fontsize=10, fontweight="bold")
        return

    labels = list(cm_dict.keys())
    matrix_size = len(labels)
    cm_normalized = np.zeros((matrix_size, matrix_size))

    for i, actual in enumerate(labels):
        for j, pred in enumerate(labels):
            cm_normalized[i, j] = cm_dict[actual].get(pred, 0.0)

    annot_labels = np.array(
        [
            [f"{val:.1f}%" if val > 0 else "0" for val in row]
            for row in cm_normalized
        ]
    )

    sns.heatmap(
        cm_normalized,
        annot=annot_labels,
        fmt="",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        cbar=False,
        vmin=0,
        vmax=100,
        ax=ax,
        annot_kws={"size": 6},
    )

    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    ax.set_xlabel("Predicted Activity", fontsize=8, fontweight="bold")
    ax.set_ylabel("Actual Activity", fontsize=8, fontweight="bold")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right", fontsize=7)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=7)


def generate_pdf_report(json_filename):
    json_file_path = os.path.join(logs_dir, json_filename)

    if not os.path.exists(json_file_path):
        print(f"Skipping: '{json_filename}' not found.")
        return

    with open(json_file_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    instances = payload.get("instances", [])
    if not instances:
        print(f"Skipping: '{json_filename}' has no experiment instances.")
        return

    now = dt.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    base_name = os.path.splitext(json_filename)[0]
    pdf_output_path = os.path.join(logs_dir, f"{base_name}_report_{now}.pdf")

    with PdfPages(pdf_output_path) as pdf:
        for instance in instances:
            instance_id = instance.get("instance_id")
            timestamp = instance.get("timestamp", "N/A")
            hyperparams = instance.get("parameters", {})
            split_config = instance.get("split_config", {})

            fig = plt.figure(figsize=(12, 8.5))
            gs = fig.add_gridspec(
                2, 3, height_ratios=[1.2, 1.0], hspace=0.40, wspace=0.28
            )

            plt.suptitle(
                f"Experiment Instance #{instance_id} Evaluation Report | Executed: {timestamp}",
                fontsize=12,
                fontweight="bold",
                y=0.97,
            )

            # Check special features present in this instance
            has_early_stopping = any(
                "best_iteration" in extract_metrics(instance.get(k))
                for k, _ in dataset_keys
            )
            has_fold_stats = any(
                "mean_fold_accuracy" in extract_metrics(instance.get(k))
                for k, _ in dataset_keys
            )

            # -------------------------------------------------------------
            # Row 1: Confusion Matrices
            # -------------------------------------------------------------
            metrics_summary = []

            for col_idx, (ds_key, ds_name) in enumerate(dataset_keys):
                ax = fig.add_subplot(gs[0, col_idx])
                ds_data = instance.get(ds_key, {})
                metrics = extract_metrics(ds_data)

                cm_dict = metrics.get("confusion_matrix_pct", {})
                acc = metrics.get("accuracy", 0.0)
                f1 = metrics.get("f1_macro", 0.0)
                precision = metrics.get("precision_macro", 0.0)
                recall = metrics.get("recall_macro", 0.0)

                row_data = [
                    ds_name,
                    f"{acc:.4f} ({acc * 100:.2f}%)",
                    f"{f1:.4f}",
                    f"{precision:.4f}",
                    f"{recall:.4f}",
                ]

                if has_early_stopping:
                    best_iter = metrics.get("best_iteration")
                    row_data.append(str(best_iter) if best_iter is not None else "N/A")

                if has_fold_stats:
                    mean_acc = metrics.get("mean_fold_accuracy", 0.0)
                    std_acc = metrics.get("std_fold_accuracy", 0.0)
                    row_data.append(f"{mean_acc:.4f} ± {std_acc:.4f}")

                metrics_summary.append(row_data)

                plot_title = f"{ds_name}\nAcc: {acc * 100:.2f}% | Macro F1: {f1:.4f}"
                render_confusion_matrix(ax, cm_dict, plot_title)

            # -------------------------------------------------------------
            # Row 2, Col 0: Combined Hyperparameters & Split Parameters Table
            # -------------------------------------------------------------
            ax_param = fig.add_subplot(gs[1, 0])
            ax_param.axis("off")

            param_table_data = []

            # Include key split configuration details if available
            if split_config:
                if "train_size_ratio" in split_config:
                    param_table_data.append(["train_ratio", str(split_config["train_size_ratio"])])
                if "val_size_ratio" in split_config:
                    param_table_data.append(["val_ratio", str(split_config["val_size_ratio"])])
                if "test_size_ratio" in split_config:
                    param_table_data.append(["test_ratio", str(split_config["test_size_ratio"])])
                if "early_stopping_rounds" in split_config:
                    param_table_data.append(["early_stopping", str(split_config["early_stopping_rounds"])])

            for k, v in hyperparams.items():
                param_table_data.append([str(k), str(v)])

            param_table = ax_param.table(
                cellText=param_table_data,
                colLabels=["Config Parameter", "Value"],
                loc="center",
                cellLoc="left",
            )
            param_table.auto_set_font_size(False)
            param_table.set_fontsize(7.5)
            param_table.scale(1.0, 1.15)

            for col_idx in range(2):
                cell = param_table[(0, col_idx)]
                cell.set_facecolor("#40466e")
                cell.get_text().set_color("white")
                cell.get_text().set_weight("bold")

            ax_param.set_title(
                "Model & Split Configuration",
                fontsize=10,
                fontweight="bold",
                pad=6,
            )

            # -------------------------------------------------------------
            # Row 2, Col 1-2: Performance Summary Table
            # -------------------------------------------------------------
            ax_summary = fig.add_subplot(gs[1, 1:])
            ax_summary.axis("off")

            summary_headers = [
                "Dataset Variant",
                "Accuracy",
                "F1 (Macro)",
                "Precision",
                "Recall",
            ]
            if has_early_stopping:
                summary_headers.append("Best Iteration")
            if has_fold_stats:
                summary_headers.append("Mean Fold Acc")

            summary_table = ax_summary.table(
                cellText=metrics_summary,
                colLabels=summary_headers,
                loc="center",
                cellLoc="center",
            )
            summary_table.auto_set_font_size(False)
            summary_table.set_fontsize(8)
            summary_table.scale(1.0, 1.4)

            for col_idx in range(len(summary_headers)):
                cell = summary_table[(0, col_idx)]
                cell.set_facecolor("#40466e")
                cell.get_text().set_color("white")
                cell.get_text().set_weight("bold")

            for row_idx in range(1, len(metrics_summary) + 1):
                bg_color = "#f2f2f2" if row_idx % 2 == 0 else "#ffffff"
                for col_idx in range(len(summary_headers)):
                    summary_table[(row_idx, col_idx)].set_facecolor(bg_color)

            ax_summary.set_title(
                "Performance Comparison Across Dataset Permutations",
                fontsize=10,
                fontweight="bold",
                pad=6,
            )

            pdf.savefig(fig)
            plt.close(fig)

    print(f"Successfully generated PDF report: {pdf_output_path}")


# Execute report generation for all specified JSON files
if __name__ == "__main__":
    for json_file in json_files_to_process:
        generate_pdf_report(json_file)

Successfully generated PDF report: ../dataset/xgb_gss_results_report_2026-10-07_21-41-25.pdf
Skipping: 'xgb_gss_val_results.json' not found.
Successfully generated PDF report: ../dataset/xgb_permutation_results_report_2026-10-07_21-41-29.pdf
